# Project - Airline AI Assistant

We'll now bring together what we've learned to make an AI Customer Support assistant for an Airline

In [3]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [4]:
load_dotenv(override=True)

openai_api_key = os.getenv('OPENROUTER_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:3]}")
else:
    print("OpenAI API Key not set")

openai_url = "https://openrouter.ai/api/v1"
model = "openai/gpt-4o-mini"
openai = OpenAI(
    base_url = openai_url,
    api_key = openai_api_key
)

OpenAI API Key exists and begins sk-


In [5]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [6]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=model, messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Tools

Tools are an incredibly powerful feature provided by the frontier LLMs.

With tools, you can write a function, and have the LLM call that function as part of its response.

Sounds almost spooky.. we're giving it the power to run code on our machine?

Well, kinda.

In [7]:
# Start by making a useful function

ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print(f"Tool called for city {destination_city}")
    price = ticket_prices.get(destination_city.lower(), "Unknown ticket price")
    return f"The price of a ticket to {destination_city} is {price}"

In [8]:
get_ticket_price("tokyo")

Tool called for city tokyo


'The price of a ticket to tokyo is $1400'

In [9]:
# There's a particular dictionary structure that's required to describe our function:

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [10]:
# And this is include in a list of tools:

tools = [{"type": "function", "function": price_function}]

In [11]:
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

## Getting OpenAI to use our Tool

There's some fiddly stuff to allow OpenAI "to call our tool"

What we actually do is give the LLM the opportunity to inform us that it wants us to run the tool.

Here's how the new chat function looks:

In [12]:
def chat(message, history): # function callback yang di panggil gradio setiap mengirim pesan
    history = [{"role":h["role"], "content":h["content"]} for h in history] # list comprehension yang membaca ulang tiap history lalu hanya ambil {"role"} dan {"content"}
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}] # menyusun daftar lengkap yang di kirim ke LLM beserta history(system_message + history + pesan user terbaru)
    response = openai.chat.completions.create(model=model, messages=messages, tools=tools) # membuat response LLM (model, jawaban AI, tools akses)

    if response.choices[0].finish_reason=="tool_calls": # cek apakah LLM minta tool di jalankan ? jika tidak ada jawaban ke user (finish_reason) sama dengan jalankan ("tool_calls") jika tidak lewati proses langsung return response
        message = response.choices[0].message # ambil object pesan dari LLM yang berisi permintaan tool(lengkap:nama function, argument, "tool_call_id"), variabel message menimpa parameter message(agak membingungkan) nama variabel yang sama untuk dua hal yang berbeda, tidak error karena parameter "message" yang lama sudah tidak di pakai lagi
        response = handle_tool_call(message) # serahkan pesan ke function yang ada di cell bawah, hasil (dict berisi harga)
        messages.append(message) # tempel/tambahkan pesan LLM (yang minta tool) ke history chat
        messages.append(response) # tempe/tambahkan hasil tool ke history berisi: system -> history -> user -> [LLM minta tool] -> [hasil tool]
        response = openai.chat.completions.create(model=model, messages=messages) # panggilan kedua ke LLM, kali ini LLM melihat history lengkap termsuk hasil tool, lalu menyusun jawaban
    return response.choices[0].message.content # kembalikan text jawaban akhir hasil dari LLM

# alur logic: panggil LLM -> kalau minta tool, jalankan tool lalu panggil LLM lagi -> kembalikan jawaban 
# ini versi if (satu kali tool), ubah jadi while supaya bisa request/menggunakan toll_calls berkali kali

In [13]:
# We have to whrite that function handle_tool_call:

def handle_tool_call(message): # terima pesan LLM yang berisi permintaan tool
    tool_call = message.tool_calls[0] # ambil request tool pertama[0]. note: code ini hanya cuma memproses request yang pertama sisanya di abaikan -> error(harga london dan paris = london)  
    if tool_call.function.name == "get_ticket_price": # cek nama function, jika get_ticket_price maka jalankan if code di dalamnya 
        arguments = json.loads(tool_call.function.arguments) # argument dari LLM sebagai json, (json_load) mengubah jadi dictionary python supaya bisa di baca 
        city = arguments.get("destination_city") # ambil value argument nya yaitu kota(london)
        price_details = get_ticket_price(city) # inilah eksekusi asli tool, mencari harga ticket
        response = { # bungkus hasilnya jadi pesan format tool
            "role": "tool", # role ke empat selain(system/user/assistant + tools) menandai hasil dari tool
            "content": price_details, # isi hasilnya (teks harga)
            "tool_call_id" : tool_call.id # penghubung id harus cocok dengan id di request tool tadi, supaya LLM tau "hasil ini untuk request yang mana"
        }
    return response # kembalikan dict cell atas lalu menempelkannya ke "message"

In [14]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


## Let's make a couple of improvements

Handling multiple tool calls in 1 response

Handling multiple tool calls 1 after another

In [15]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=model, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        response = handle_tool_calls(message)
        messages.append(message)
        messages.extend(response) # mengembalikan list of dicts(bisa banyak hasil tool) jadi pakai extend -- membongkar list dan menempelkannya tiap item satu per satu (hasil tool) + mesages, (hasil tool) + messages
        response = openai.chat.completions.create(model=model, messages=messages)
    return response.choices[0].message.content

# messages = ["a", "b"]
# data = [1, 2, 3]

# messages.append(data)   # → ["a", "b", [1, 2, 3]]   ← list nyangkut jadi 1 elemen (SALAH)
# messages.extend(data)   # → ["a", "b", 1, 2, 3]     ← tiap item masuk terpisah (BENAR)

In [16]:
def handle_tool_calls(message): # terima pesan LLM yang request tool, bisa loop semua tool 
    responses = [] # siapkan wadah kosong [list]
    for tool_call in message.tool_calls: # loop setiap permintaan tool
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get("destination_city")
            price_details = get_ticket_price(city)
            responses.append({ # kumpulkan tiap hasil ke responses wadah [list]
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses # kembalikan LIST berisi semua hasil

In [17]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


In [18]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=model, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=model, messages=messages, tools=tools) # tools=tools di pakai di call ke dua karena setiap call ke LLM itu stateless-- akses ke tool tidak terbawa dari call sebelumnya, harus di sertakan ulang setiap kali karena menggunakan looping while
    return response.choices[0].message.content


In [ ]:
# modul database bawaan python
# SQlite menyimpan seluruh database dalam satu file di lokal, tanpa server terpisah

import sqlite3

In [ ]:
# with = untuk mengelola resource yang perlu di "buka" lalu di "tutup" otomatis seperti koneksi database atau open("data.txt") as f, isi = f.read()
# conn = singkatan dari connection, adalah object yang mewakili antara program dengan file database

DB = "prices.db" # nama file database, jika belum ada SQlite otomatis membuatkannya

with sqlite3.connect(DB) as conn: # buka koneksi ke database price.db(DB) dan mengembalikan ke object conn
    cursor = conn.cursor() # cursor untuk menjalankan perintah SQL dan membaca hasilnya
    cursor.execute("CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)") # jalankan perintah SQL untuk membuat tabel, di baca bagian per bagian:
    conn.commit() # simpan permanen perubahan ke file tanpa commit, perubahan hanya tersimpan sementara 

In [21]:
get_ticket_price("london")

Tool called for city london


'The price of a ticket to london is $799'

In [29]:
def set_ticket_price(city, price): # function untuk menyimpan harga ke database
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?", (city.lower(), price, price)) # masukan baris baru ke tabel princes dengan nilai untuk kolom city dan prince
        conn.commit()

In [27]:
ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999} # dictionary awal harga
for city, price in ticket_price.items(): # loop tiap pasangan key:value (london:799)
    set_ticket_price(city, price) # call function di atas untuk menyimpan tiap kota ke database

In [28]:
get_ticket_price("tokyo") # database sudah terisi, mengembalikan harga tokyo

Tool called for city tokyo


'The price of a ticket to tokyo is 1420'

In [30]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


Tool called for city Tokyo
